# 02 · Silver: cleaned and classified ads

Reads both bronze sources, flattens them into columns, deduplicates, classifies each ad into a data role, and writes the result to `lh_silver.dbo.ads`.

- **Input (`lh_bronze`):**
  - `Files/bronze/historical/term=…/year=…/month=…/ads.json`: historical backfill (`nb_01`)
  - `Files/bronze/jobsearch/snapshot_date=…/term=…/ads.json`: daily snapshots of open ads (`nb_05`)
- **Output:** `lh_silver.dbo.ads`: one row per ad
- **How to run:** *Run all*, or from the daily pipeline after `nb_05`. The table is overwritten, so re-running gives the same result.
- **Lakehouses:** `lh_bronze` must be the default (the notebook reads relative paths); `lh_silver` must also be attached.

## Design notes

- **Two sources, one table.** Completed months come from the Historical Ads API; recent and open ads come from daily JobSearch snapshots. Both are queried with the same search terms and return the same ad format, so they are flattened the same way and unioned.
- **When an ad appears in both, the newest version wins.** Rows are ranked by snapshot date (historical rows have none and rank last), so an ad edited by the employer is shown as it currently reads.
- **Snapshots give an observed lifetime.** `first_seen` and `last_seen` are the first and last snapshot an ad appeared in, and `is_open` marks ads present in the latest snapshot. This only covers ads seen since the daily pipeline started (2026-11-03).
- **Incremental ingestion, full-refresh transformation.** Bronze only ever adds files; silver is rebuilt from all of bronze on every run. At ~12,000 ads this takes a few minutes, and it means a change to the classification rules applies to the whole history at once.
- **Known gap:** the historical backfill ends in August 2026, and the first snapshot only contains ads still open on 2026-11-03. Ads published in September–October that had already closed are missing until the historical API catches up and the backfill is extended.
- **Search terms decide what was fetched, not what an ad is.** The API expands a term into its own concepts: *BI developer* became the skill "bi" plus the occupation "systems developer", which pulled in full-stack developers and even a laboratory engineer. The role is therefore decided here, from the ad headline.
- **Occupation codes are not reliable either.** Ads titled "Data Analyst" carry occupation labels such as *Database developer*, *Systems analyst* and *Public administration officer*, and BI developer has no occupation code at all.
- **Headlines are free text**, so matching is done on a normalised headline (lower case, punctuation replaced by spaces) with word-boundary patterns: `BI- utvecklare`, `BI/DW Utvecklare` and `BI MICROSOFT DEVELOPER` all have to match, while `bibliotek` and `Data Centre Engineering` must not.
- **Rules are ordered from specific to general.** *Analytics Engineer* is tested before *Data Engineer*, otherwise it would never match.
- **Excluded: "Online Data Analyst".** 177 historical ads (almost all from Jobs Europe AB, 2023–2024) are remote gig jobs rating search results in a given language, not data analysis. They made up about 20% of Data Analyst ads in 2023–2024 and mention few tools, so they both inflated the role count and diluted its skill shares. Removing them changed the picture for the role: Data Analyst ads peaked in 2022 and have declined since, rather than levelling off. They were found while checking ads without a location, not during the original classification.
- **Ads that match no rule are dropped.** In the historical data, 1,662 of 13,380 unique ads matched no role rule (mostly from the two noisy search terms) and 177 were excluded as above. Bronze keeps everything, so the scope can be changed by re-running this notebook.
- **Overlap between search terms is kept as information.** About 19% of historical ad rows were found via more than one term; the set of terms is stored in `search_terms`.
- **Known limitations of the location fields.** The historical data for 2016–2017 contains no workplace address, so location analysis should start in 2018. Staffing agencies sometimes give their own office as the location.
- **Accuracy:** in a manual sample of 30 classified ads, 1–2 looked wrong, roughly 95% precision on a small sample. The sample did not catch the "Online Data Analyst" problem, which affected 1.5% of all ads.


## 1. Setup

In [1]:
from pyspark.sql import functions as F, Window as W

HISTORICAL_PATH = "Files/bronze/historical"   # relative to the default lakehouse (lh_bronze)
JOBSEARCH_PATH = "Files/bronze/jobsearch"
SILVER_TABLE = "lh_silver.dbo.ads"

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 3, Finished, Available, Finished, False)

## 2. Read and flatten both sources

Spark reads the `key=value` folder names as columns. `multiLine` is needed because each file is one JSON object. Each source is flattened to the same columns before the union, because the two sources' inferred schemas differ slightly (newer ads have fields older ones lack).

In [2]:
def flatten(raw, source):
    """One row per ad and search term, with the fields the analysis needs."""
    snapshot = (F.col("snapshot_date").cast("date") if "snapshot_date" in raw.columns
                else F.lit(None).cast("date"))
    return (raw
        .select(F.col("_meta.search_term").alias("search_term"),
                snapshot.alias("snapshot_date"),
                F.explode("hits").alias("ad"))
        .select(
            F.lit(source).alias("source"),
            "search_term",
            "snapshot_date",
            F.col("ad.id").alias("ad_id"),
            F.col("ad.headline").alias("headline"),
            F.col("ad.description.text").alias("description"),
            F.col("ad.employer.name").alias("employer_name"),
            F.col("ad.employer.organization_number").alias("employer_orgnr"),
            F.col("ad.workplace_address.region").alias("region"),
            F.col("ad.workplace_address.municipality").alias("municipality"),
            F.col("ad.occupation.label").alias("occupation"),
            F.col("ad.occupation_group.label").alias("occupation_group"),
            F.col("ad.occupation_field.label").alias("occupation_field"),
            F.col("ad.employment_type.label").alias("employment_type"),
            F.col("ad.working_hours_type.label").alias("working_hours_type"),
            F.col("ad.number_of_vacancies").alias("n_vacancies"),
            F.to_timestamp("ad.publication_date").alias("published_at"),
            F.to_timestamp("ad.last_publication_date").alias("last_published_at"),
            F.to_timestamp("ad.application_deadline").alias("application_deadline"),
            F.col("ad.removed").alias("removed"),
            F.col("ad.webpage_url").alias("webpage_url"),
        ))

read = lambda path: spark.read.option("multiLine", "true").json(path)
historical = flatten(read(HISTORICAL_PATH), "historical")
jobsearch = flatten(read(JOBSEARCH_PATH), "jobsearch")
rows = historical.unionByName(jobsearch)

rows.groupBy("source").agg(F.count("*").alias("ad_rows"),
                           F.countDistinct("ad_id").alias("unique_ads"),
                           F.countDistinct("snapshot_date").alias("snapshots")).show()
print("Unique ads across both sources:", rows.select("ad_id").distinct().count())

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 4, Finished, Available, Finished, False)

+----------+-------+----------+---------+
|    source|ad_rows|unique_ads|snapshots|
+----------+-------+----------+---------+
| jobsearch|    216|       175|        1|
|historical|  16458|     13380|        0|
+----------+-------+----------+---------+

Unique ads across both sources: 13555


## 3. Per-ad information that spans rows

Collected before deduplication: which search terms found the ad, and in which snapshots it was seen.

In [9]:
latest_snapshot = jobsearch.agg(F.max("snapshot_date")).first()[0]
print("Latest snapshot:", latest_snapshot)

per_ad = (rows.groupBy("ad_id").agg(
        F.sort_array(F.collect_set("search_term")).alias("search_terms"),
        F.min("snapshot_date").alias("first_seen"),
        F.max("snapshot_date").alias("last_seen"))
    .withColumn("n_terms", F.size("search_terms"))
    .withColumn("is_open", F.coalesce(F.col("last_seen") == F.lit(latest_snapshot), F.lit(False))))

per_ad.groupBy("n_terms").count().orderBy("n_terms").show()

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 11, Finished, Available, Finished, False)

Latest snapshot: 2026-11-03
+-------+-----+
|n_terms|count|
+-------+-----+
|      1|12413|
|      2| 1097|
|      3|   44|
|      4|    1|
+-------+-----+



## 4. Deduplicate: keep the newest version of each ad

In [4]:
newest_first = W.partitionBy("ad_id").orderBy(F.col("snapshot_date").desc_nulls_last())

flat = (rows
    .withColumn("rn", F.row_number().over(newest_first))
    .filter("rn = 1")
    .drop("rn", "search_term", "snapshot_date")
    .join(per_ad, "ad_id", "left"))

print("Rows after dedup:", flat.count())
flat.groupBy("source").count().show()

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 6, Finished, Available, Finished, False)

Rows after dedup: 13555
+----------+-----+
|    source|count|
+----------+-----+
| jobsearch|  175|
|historical|13380|
+----------+-----+



## 5. Classify into roles

Rules are applied top-down on a normalised headline. The first rule is an exclusion.

In [5]:
# "BI- utvecklare" -> "bi utvecklare": lower case, any punctuation becomes a space.
norm = F.regexp_replace(F.lower(F.col("headline")), r"[^a-z0-9åäö]+", " ")

classified = (flat
    .withColumn("headline_norm", norm)
    .withColumn("role",
        # Excluded: search-rating gig jobs titled "Online Data Analyst" (see design notes)
        F.when(F.col("headline_norm").rlike(r"online data analyst"), F.lit(None))
         .when(F.col("headline_norm").rlike(r"analytics engineer"), "Analytics Engineer")
         .when(F.col("headline_norm").rlike(r"\b(data|big data|ml|machine learning)\b.*\bengineer\b|dataingenjör|datatekniker"), "Data Engineer")
         .when(F.col("headline_norm").rlike(r"data scientist|datavetare|machine learning"), "Data Scientist")
         .when(F.col("headline_norm").rlike(r"data analyst|dataanalytiker|dataanalys|\bdata analys"), "Data Analyst")
         .when(F.col("headline_norm").rlike(r"\bbi\b|\bdw\b|business intelligence|power bi|qlik|data warehouse|datalager"), "BI Developer")
         .otherwise(F.lit(None))))

classified.groupBy("role").count().orderBy(F.desc("count")).show(truncate=False)

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 7, Finished, Available, Finished, False)

+------------------+-----+
|role              |count|
+------------------+-----+
|Data Engineer     |4527 |
|BI Developer      |2375 |
|Data Analyst      |2327 |
|Data Scientist    |2216 |
|NULL              |1882 |
|Analytics Engineer|228  |
+------------------+-----+



## 6. Inspect the classification

Check both directions: what the rules miss, and whether what they catch is right. The sample is meant to be read by a human.

In [6]:
print("Unclassified:", classified.filter("role is null").count())
classified.filter("role is null").select("headline", "search_terms").show(20, truncate=60)

classified.filter("role is not null").select("role", "headline").sample(0.02, seed=42).show(30, truncate=60)

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 8, Finished, Available, Finished, False)

Unclassified: 1882
+------------------------------------------------------------+-----------------------------+
|                                                    headline|                 search_terms|
+------------------------------------------------------------+-----------------------------+
|Kvantitativ analytiker för analys av hälsa och läkemedels...|             [data scientist]|
| Laboratorieingenjör till Octapharma, Kungsholmen, Stockholm|       ["analytics engineer"]|
|                                          Fullstack engineer|[BI developer, data engineer]|
|                          Utvecklare till Beijer Electronics|               [BI developer]|
|                               Digital Product Manager, Data|             [dataanalytiker]|
|    Systemutvecklare till Swedemount Sportswear & Fashion AB|               [BI developer]|
|                 Full stack developer with focus on frontend|               [BI developer]|
|         Applikationsutvecklare till vår kund på K

## 7. Write the silver table

In [7]:
silver = (classified
    .filter("role is not null")
    .drop("headline_norm")
    .withColumn("ingested_at", F.current_timestamp()))

(silver.write
    .mode("overwrite")                      # idempotent: re-running replaces the table
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print("Rows written:", spark.table(SILVER_TABLE).count())

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 9, Finished, Available, Finished, False)

Rows written: 11673


## 8. Verify

In [8]:
t = spark.table(SILVER_TABLE)

print("Ads per role and year (current year is partial):")
(t.groupBy(F.year("published_at").alias("year"), "role").count()
  .groupBy("year").pivot("role").sum("count").orderBy("year").show())

t.groupBy("source").agg(F.count("*").alias("ads"),
                        F.sum(F.col("is_open").cast("int")).alias("open_now")).show()

n_rows = t.count()
n_ids = t.select("ad_id").distinct().count()
n_no_role = t.filter("role is null").count()
n_no_date = t.filter("published_at is null").count()
n_no_text = t.filter("description is null or length(description) = 0").count()
print(f"Rows: {n_rows}, unique ids: {n_ids}, missing role: {n_no_role}, "
      f"missing publication date: {n_no_date}, missing description: {n_no_text}")

assert n_rows == n_ids, "Duplicate ad ids in the silver table"
assert n_no_role == 0, "Unclassified ads were written"
assert n_no_date == 0, "Ads without a publication date"

StatementMeta(, 7b1a02a5-1602-48fd-90f0-d9249bcc80b8, 10, Finished, Available, Finished, False)

Ads per role and year (current year is partial):
+----+------------------+------------+------------+-------------+--------------+
|year|Analytics Engineer|BI Developer|Data Analyst|Data Engineer|Data Scientist|
+----+------------------+------------+------------+-------------+--------------+
|2016|                 5|         123|          37|           30|            60|
|2017|                 7|         157|          63|           38|            86|
|2018|              NULL|         189|          99|          117|           151|
|2019|                 2|         270|         147|          207|           205|
|2020|                 4|         187|         139|          242|           198|
|2021|                27|         252|         305|          548|           319|
|2022|                23|         303|         370|          677|           297|
|2023|                40|         253|         347|          647|           253|
|2024|                31|         267|         321|         